In [16]:
!pip install -q google-genai

In [17]:
from google.colab import userdata
import google.generativeai as genai
import pandas as pd
import json
from datetime import datetime

# Load the key safely
api_key = userdata.get('GEMINI_API_KEY')

if not api_key:
    raise ValueError("❌ Secret 'GEMINI_API_KEY' not found. Check the name and Notebook access.")

genai.configure(api_key=api_key)
print("✅ API key loaded successfully from Secrets")

✅ API key loaded successfully from Secrets


In [18]:
from google.colab import userdata
from google import genai
from google.genai import types
import pandas as pd
import json
from datetime import datetime

api_key = userdata.get('GEMINI_API_KEY')

if not api_key:
    raise ValueError("❌ Secret 'GEMINI_API_KEY' not found. Check the name and Notebook access.")

client = genai.Client(api_key=api_key)
print("✅ New Google GenAI client created successfully")

MODEL_NAME = "gemini-2.0-flash"

✅ New Google GenAI client created successfully


In [19]:
# Free models that usually work well
MODEL_NAME = "gemini-2.0-flash"          # Try this first
# MODEL_NAME = "gemini-1.5-flash"        # Backup option
# MODEL_NAME = "gemini-3.8-flash"        # Another good free option

print(f"Using model: {MODEL_NAME}")

Using model: gemini-2.0-flash


Sample Alerts

In [20]:
alerts = [
    {
        "id": "ALT-001",
        "timestamp": "2026-11-17T22:14:03Z",
        "source": "Identity Provider",
        "type": "failed_login",
        "raw_details": "15 failed login attempts for user 'admin' from IP 185.220.101.45 (Netherlands) within 4 minutes. Account is privileged."
    },
    {
        "id": "ALT-002",
        "timestamp": "2026-11-17T23:01:18Z",
        "source": "EDR",
        "type": "malware_detected",
        "raw_details": "EDR detected 'emotet' variant on workstation WS-HR-042. File: C:\\Users\\j.smith\\Downloads\\invoice_update.exe. Process was quarantined."
    },
    {
        "id": "ALT-003",
        "timestamp": "2026-11-18T01:33:47Z",
        "source": "Firewall",
        "type": "suspicious_ip",
        "raw_details": "Outbound connection from internal host 10.0.12.88 to known C2 IP 91.219.236.171 on port 443. Connection lasted 47 seconds."
    },
    {
        "id": "ALT-004",
        "timestamp": "2026-11-18T02:15:09Z",
        "source": "Email Gateway",
        "type": "phishing_attempt",
        "raw_details": "User 'm.lopez' clicked a link in an email claiming to be from IT Support. Destination domain: secure-login-update[.]xyz. No credentials entered yet."
    },
    {
        "id": "ALT-005",
        "timestamp": "2026-11-18T03:42:11Z",
        "source": "Cloud Workload",
        "type": "unusual_api_call",
        "raw_details": "Service account 'data-pipeline-sa' made 120 ListBuckets calls in 3 minutes from an unusual region (ap-southeast-1). Normal activity is only in us-east-1."
    },
    {
        "id": "ALT-006",
        "timestamp": "2026-11-18T04:05:33Z",
        "source": "Endpoint",
        "type": "privilege_escalation",
        "raw_details": "User 'temp.contractor' successfully added themselves to the local Administrators group on laptop LT-FIN-019 using net localgroup."
    }
]

print(f"✅ Loaded {len(alerts)} sample alerts")

✅ Loaded 6 sample alerts


System Prompt + Triage Function

In [21]:
SYSTEM_PROMPT = """
You are a Tier-1 SOC analyst working in a Security Operations Center.
Your job is to perform first-level triage on security alerts.

For every alert you receive, you must return ONLY a valid JSON object with these exact fields:

{
  "severity": "Critical" | "High" | "Medium" | "Low" | "Informational",
  "summary": "One clear sentence describing what happened",
  "likely_attack_type": "Short name of the most probable attack or activity",
  "recommended_action": "What the analyst should do next (1-2 concrete steps)",
  "confidence": "High" | "Medium" | "Low"
}

Rules:
- Be concise and professional.
- Do not invent information that is not in the alert.
- Prefer higher severity when the alert involves privileged accounts, confirmed malware, or C2 communication.
- Return ONLY the JSON object. No markdown, no extra text, no explanations.
"""

def triage_alert(alert: dict) -> dict:
    user_prompt = f"""
Alert ID: {alert['id']}
Timestamp: {alert['timestamp']}
Source: {alert['source']}
Type: {alert['type']}

Raw Details:
{alert['raw_details']}

Perform triage and return the JSON object only.
"""

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                temperature=0.2,
                response_mime_type="application/json"
            )
        )

        result = json.loads(response.text)

        result["alert_id"] = alert["id"]
        result["original_type"] = alert["type"]
        result["source"] = alert["source"]
        result["timestamp"] = alert["timestamp"]

        return result

    except Exception as e:
        print(f"❌ Error on {alert['id']}: {type(e).__name__} → {e}")
        return {
            "alert_id": alert["id"],
            "severity": "Error",
            "summary": str(e),
            "likely_attack_type": "N/A",
            "recommended_action": "Manual review required",
            "confidence": "Low",
            "original_type": alert["type"],
            "source": alert["source"],
            "timestamp": alert["timestamp"]
        }

Run Triage

In [22]:
print("🚀 Starting triage...\n")

results = []

for alert in alerts:
    print(f"Processing {alert['id']} ({alert['type']})...")
    triage = triage_alert(alert)
    results.append(triage)
    print(f"   → Severity: {triage.get('severity')} | {triage.get('likely_attack_type')}\n")

print("✅ Triage completed!")

🚀 Starting triage...

Processing ALT-001 (failed_login)...
❌ Error on ALT-001: LocalProtocolError → Illegal header value b"from google.colab import userdata\r\nuserdata.get('secretName')"
   → Severity: Error | N/A

Processing ALT-002 (malware_detected)...
❌ Error on ALT-002: LocalProtocolError → Illegal header value b"from google.colab import userdata\r\nuserdata.get('secretName')"
   → Severity: Error | N/A

Processing ALT-003 (suspicious_ip)...
❌ Error on ALT-003: LocalProtocolError → Illegal header value b"from google.colab import userdata\r\nuserdata.get('secretName')"
   → Severity: Error | N/A

Processing ALT-004 (phishing_attempt)...
❌ Error on ALT-004: LocalProtocolError → Illegal header value b"from google.colab import userdata\r\nuserdata.get('secretName')"
   → Severity: Error | N/A

Processing ALT-005 (unusual_api_call)...
❌ Error on ALT-005: LocalProtocolError → Illegal header value b"from google.colab import userdata\r\nuserdata.get('secretName')"
   → Severity: Error | 

In [23]:
df = pd.DataFrame(results)

# Nice column order
column_order = [
    "alert_id", "timestamp", "source", "original_type",
    "severity", "summary", "likely_attack_type",
    "recommended_action", "confidence"
]

df = df[column_order]

# Show the table
display(df)

# Save CSV
csv_filename = f"security_alert_triage_report_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
df.to_csv(csv_filename, index=False)

print(f"\n📁 Report saved as: {csv_filename}")
print("You can download it from the Files panel on the left.")

,alert_id,timestamp,source,original_type,severity,summary,likely_attack_type,recommended_action,confidence
0,ALT-001,2026-11-17T22:14:03Z,Identity Provider,failed_login,Error,"Illegal header value b""from google.colab impor...",N/A,Manual review required,Low
1,ALT-002,2026-11-17T23:01:18Z,EDR,malware_detected,Error,"Illegal header value b""from google.colab impor...",N/A,Manual review required,Low
2,ALT-003,2026-11-18T01:33:47Z,Firewall,suspicious_ip,Error,"Illegal header value b""from google.colab impor...",N/A,Manual review required,Low
3,ALT-004,2026-11-18T02:15:09Z,Email Gateway,phishing_attempt,Error,"Illegal header value b""from google.colab impor...",N/A,Manual review required,Low
4,ALT-005,2026-11-18T03:42:11Z,Cloud Workload,unusual_api_call,Error,"Illegal header value b""from google.colab impor...",N/A,Manual review required,Low
5,ALT-006,2026-11-18T04:05:33Z,Endpoint,privilege_escalation,Error,"Illegal header value b""from google.colab impor...",N/A,Manual review required,Low



📁 Report saved as: security_alert_triage_report_20261122_1124.csv
You can download it from the Files panel on the left.
